# Analyse


In [1]:
from pathlib import Path
import os
import time

import numpy as np
import math
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scikit_posthocs as sp

from imfcsoutputhandlerlib.core.all_image import AllImage
from imfcsoutputhandlerlib.core.image_queries import (
    get_total_num_cell,
    get_array_intensity,
    get_array_parameter_map,
    get_list_of_file_id,
    get_cfs_related,
    get_lagtimes,
)

In [2]:
DIRNAME = "input/input_mef-dii-oa-37c/2x2-overlap"
DIRNAME_OUTPUT = "output/output_da1"

PICKLE_DATABASE_FILENAME = None  # "XX.pkl"
IS_SAVE_PLOT = False
IS_SAVE_CSV = False

In [3]:
current_dir = Path().resolve()
input_path = Path(os.path.join(current_dir.parent, DIRNAME))
output_path = Path(os.path.join(current_dir.parent, DIRNAME_OUTPUT))

In [4]:
if PICKLE_DATABASE_FILENAME is None:
    pkl_files = list(input_path.glob("*.pkl"))

    if not pkl_files:
        raise FileNotFoundError("No .pkl file found")
    if len(pkl_files) > 1:
        raise RuntimeError(f"Multiple .pkl files found: {pkl_files}")

    pkl_path = pkl_files[0]
    PICKLE_DATABASE_FILENAME = pkl_path.name
    print(f"Found: {PICKLE_DATABASE_FILENAME}")

Found: all-image-database_20260114_132958.pkl


Load database


In [5]:
time_start = time.time()
all_image_instance = AllImage.from_pickle(
    Path(os.path.join(input_path, PICKLE_DATABASE_FILENAME))
)
time_elapsed = time.time() - time_start
print(f"time taken to load database: {time_elapsed:.3f} s")

Missing 'new_method'. Added default implementation.
time taken to load database: 0.673 s


In [6]:
all_image_instance.get_list_of_image()[0]

ImageInfo(key=tirfmh29_251002_02_w1_cell1_MEF-WT_DiI-100nM-10min_15min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome, associated_files=['tirfmh29_251002_02_w1_cell1_MEF-WT_DiI-100nM-10min_15min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome_AVR.tif', 'tirfmh29_251002_02_w1_cell1_MEF-WT_DiI-100nM-10min_15min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome_bg-2x2-overlap.xlsx'], rect_coordinates=None, is_roi_selected=None)

In [7]:
time_start = time.time()
array_average_intensity_all = get_array_intensity(all_image_instance)
array_parameter_map_all = get_array_parameter_map(all_image_instance)
list_basename_all = get_list_of_file_id(all_image_instance)
array_lagtimes = get_lagtimes(all_image_instance)
array_acf1 = get_cfs_related(all_image_instance, var="acf1")
array_sd1 = get_cfs_related(all_image_instance, var="sd1")
array_fit1 = get_cfs_related(all_image_instance, var="fit1")

time_elapsed = time.time() - time_start
print(f"time loading array from AllImage instance: {time_elapsed:.2f} s")

time loading array from AllImage instance: 1.86 s


In [8]:
print(f"Total number of cells: {get_total_num_cell(all_image_instance)}")
print(
    f"shape intensity single cell: {get_array_intensity(all_image_instance, 0).shape}"
)
print(f"shape intensity all cell: {array_average_intensity_all.shape}")
print(
    f"all key {len(get_list_of_file_id(all_image_instance))}: {get_list_of_file_id(all_image_instance)}"
)
print(f"get p map single cell: {get_array_parameter_map(all_image_instance, 0).shape}")
print(f"get pmap all cell: {array_parameter_map_all.shape}")
print(f"get acf1 all cell: {array_acf1.shape}")
print(f"get sd1 all cell: {array_sd1.shape}")
print(f"get fit1 all cell: {array_fit1.shape}")
print(f"get lagtimes all cell: {array_lagtimes.shape}")

Total number of cells: 32
shape intensity single cell: (400, 95, 255, 1)
shape intensity all cell: (32, 400, 95, 255)
all key 32: ['tirfmh29_251002_02_w1_cell1_MEF-WT_DiI-100nM-10min_15min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome', 'tirfmh29_251002_05_w1_cell2_MEF-WT_DiI-100nM-10min_30min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome', 'tirfmh29_251002_08_w1_cell3_MEF-WT_DiI-100nM-10min_45min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome', 'tirfmh29_251002_11_w1_cell4_MEF-WT_DiI-100nM-10min_52min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome', 'tirfmh29_251002_14_w2_cell1_MEF-WT_DiI-100nM-10min_OA-15min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome', 'tirfmh29_251002_17_w2_cell2_MEF-WT_DiI-100nM-10min_OA-30min_561nm_590uW_pTIRF_p30_a71d34_bin1_x128y208w256h96_40k_fps238d1_1_point-TIRF-561.ome',

Set threshold for reasonable diffusion coefficient and region for analysis


In [9]:
CENTRAL_FRACTION = 0.3
D_THRESHOLD = (0.02, 20)

Show TIRF images
